# Iceberg 기반 주가 메달리온 파이프라인

yfinance에서 수집한 일봉·1분봉 데이터를 Bronze에 이력으로 보존하고,
`MERGE INTO`로 Silver 최신 상태를 만드는 학습용 파이프라인입니다.

## Goal

```text
yfinance 수집 결과
  → Bronze: 원본 수집 이력 append
  → Silver: 업무 키별 최신 레코드 MERGE
  → Gold: 분석 요구가 생길 때 확장
```

이 노트북은 테이블 생성과 Bronze→Silver 변환을 담당합니다.
실제 수집기는 Bronze 테이블에 별도로 append해야 합니다.

## Setup

- Docker Compose 스택이 실행 중이어야 합니다.
- `configs/spark/spark-defaults.conf`가 `lakehouse` REST Catalog와
  MinIO warehouse를 설정합니다.
- 모든 시각은 UTC로 해석합니다.

In [ ]:
from pyspark.sql import SparkSession

CATALOG = "lakehouse"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

spark = (
    SparkSession.builder
    .appName("iceberg-price-medallion-pipeline")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Steps

### 1. 메달리온 네임스페이스 준비

In [ ]:
for namespace in ("bronze", "silver", "gold"):
    spark.sql(
        f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{namespace}"
    )

spark.sql(f"SHOW NAMESPACES IN {CATALOG}").show(truncate=False)

### 2. Bronze 테이블 생성

Bronze는 같은 캔들이 재수집되어도 삭제하지 않습니다.
`collected_at`, `source`, `run_id`가 수집 이력을 설명합니다.

| 테이블 | 한 행의 의미 | 파티션 |
|---|---|---|
| `daily_prices_raw` | 한 수집 실행에서 관찰한 일봉 | 수집일 |
| `minute_prices_raw` | 한 수집 실행에서 관찰한 1분봉 | 캔들 거래일 |

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE}.daily_prices_raw (
    symbol STRING NOT NULL COMMENT '수집 요청 티커',
    date DATE NOT NULL COMMENT '거래일',
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT NOT NULL,
    collected_at TIMESTAMP NOT NULL COMMENT '수집 시각, UTC',
    source STRING NOT NULL COMMENT '데이터 출처',
    run_id STRING NOT NULL COMMENT '수집 실행 식별자'
)
USING iceberg
PARTITIONED BY (days(collected_at))
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
""")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE}.minute_prices_raw (
    symbol STRING NOT NULL COMMENT '티커 심볼',
    candle_time TIMESTAMP NOT NULL COMMENT '캔들 시작 시각, UTC',
    interval STRING NOT NULL COMMENT '캔들 간격',
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT,
    collected_at TIMESTAMP NOT NULL COMMENT '수집 시각, UTC',
    source STRING NOT NULL COMMENT '데이터 출처',
    run_id STRING NOT NULL COMMENT '수집 실행 식별자'
)
USING iceberg
PARTITIONED BY (days(candle_time))
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
""")

### 3. Silver 테이블 생성

Silver는 조회와 분석에 사용할 최신 상태입니다.

- 일봉 업무 키: `(symbol, date)`
- 1분봉 업무 키: `(symbol, interval, candle_time, source)`

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}.daily_prices (
    symbol STRING NOT NULL COMMENT '티커 심볼',
    date DATE NOT NULL COMMENT '거래일',
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT NOT NULL
)
USING iceberg
PARTITIONED BY (symbol)
COMMENT '정제된 일봉 주가 데이터 (중복 제거)'
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
""")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}.minute_prices (
    symbol STRING NOT NULL,
    candle_time TIMESTAMP NOT NULL COMMENT 'UTC 캔들 시작 시각',
    interval STRING NOT NULL,
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT,
    source STRING NOT NULL,
    collected_at TIMESTAMP NOT NULL
)
USING iceberg
PARTITIONED BY (days(candle_time))
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
""")

### 4. 일봉 Bronze → Silver

`ROW_NUMBER()`로 같은 종목·거래일의 가장 최근 수집분을 선택합니다.
Silver에 키가 있으면 갱신하고, 없으면 삽입합니다.

In [ ]:
spark.sql(f"""
MERGE INTO {SILVER}.daily_prices AS target
USING (
    SELECT symbol, date, open, high, low, close, volume
    FROM (
        SELECT
            symbol,
            date,
            open,
            high,
            low,
            close,
            volume,
            ROW_NUMBER() OVER (
                PARTITION BY symbol, date
                ORDER BY collected_at DESC
            ) AS row_num
        FROM {BRONZE}.daily_prices_raw
    )
    WHERE row_num = 1
) AS source
ON  target.symbol = source.symbol
AND target.date = source.date
WHEN MATCHED THEN UPDATE SET
    target.open = source.open,
    target.high = source.high,
    target.low = source.low,
    target.close = source.close,
    target.volume = source.volume
WHEN NOT MATCHED THEN INSERT (
    symbol, date, open, high, low, close, volume
) VALUES (
    source.symbol, source.date, source.open, source.high,
    source.low, source.close, source.volume
)
""")

### 5. 1분봉 Bronze → Silver

수집 출처까지 업무 키에 포함하고, 더 최신인 `collected_at`만 갱신해
오래된 재전송이 최신 값을 덮지 못하게 합니다.

In [ ]:
spark.sql(f"""
MERGE INTO {SILVER}.minute_prices AS target
USING (
    SELECT
        symbol,
        candle_time,
        interval,
        open,
        high,
        low,
        close,
        volume,
        source,
        collected_at
    FROM (
        SELECT
            symbol,
            candle_time,
            interval,
            open,
            high,
            low,
            close,
            volume,
            source,
            collected_at,
            ROW_NUMBER() OVER (
                PARTITION BY
                    symbol, interval, candle_time, source
                ORDER BY collected_at DESC
            ) AS row_num
        FROM {BRONZE}.minute_prices_raw
    )
    WHERE row_num = 1
) AS source
ON  target.symbol = source.symbol
AND target.interval = source.interval
AND target.candle_time = source.candle_time
AND target.source = source.source
WHEN MATCHED
  AND source.collected_at > target.collected_at
THEN UPDATE SET
    target.open = source.open,
    target.high = source.high,
    target.low = source.low,
    target.close = source.close,
    target.volume = source.volume,
    target.collected_at = source.collected_at
WHEN NOT MATCHED THEN INSERT (
    symbol,
    candle_time,
    interval,
    open,
    high,
    low,
    close,
    volume,
    source,
    collected_at
) VALUES (
    source.symbol,
    source.candle_time,
    source.interval,
    source.open,
    source.high,
    source.low,
    source.close,
    source.volume,
    source.source,
    source.collected_at
)
""")

## Checks

### 테이블별 행 수

In [ ]:
spark.sql(f"""
SELECT 'bronze.daily_prices_raw' AS table_name, COUNT(*) AS row_count
FROM {BRONZE}.daily_prices_raw
UNION ALL
SELECT 'silver.daily_prices', COUNT(*)
FROM {SILVER}.daily_prices
UNION ALL
SELECT 'bronze.minute_prices_raw', COUNT(*)
FROM {BRONZE}.minute_prices_raw
UNION ALL
SELECT 'silver.minute_prices', COUNT(*)
FROM {SILVER}.minute_prices
""").show(truncate=False)

### Silver 1분봉 유일성과 OHLC 규칙

In [ ]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT struct(
        symbol, interval, candle_time, source
    )) AS distinct_business_keys,
    SUM(CASE
        WHEN high < greatest(open, close, low)
          OR low > least(open, close, high)
        THEN 1 ELSE 0
    END) AS invalid_ohlc_rows
FROM {SILVER}.minute_prices
""").show(truncate=False)

## Next Steps

- Bronze 적재는 별도 수집 작업에서 append 방식으로 수행합니다.
- `minute_price_quality_checks.ipynb`에서 Bronze/Silver 정합성과
  Iceberg 스냅샷을 읽기 전용으로 점검합니다.
- 데이터 규모와 조회 패턴이 확인되면 일봉 파티션 전략과
  작은 파일 compaction 주기를 조정합니다.